# Setup

In [ ]:
import torch
import numpy as np
import config
import matplotlib.pyplot as plt
import gc
from preprocess import year_mask
from vae_train import VAEConv3D
from vae_analysis import make_analysis_loader, reconstruct_data, calculate_r2_score
from vae_analysis import fit_latent_pca, plot_scree, compute_latent_tsne, plot_tsne_by_split, plot_tsne_by_season


DATA = "/file/pathway"
SURFACE = ["msl", "t2m", "r", "u10", "v10", "tcc", "ssr"]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X = np.load(f"{DATA}/samples.npy", mmap_mode="r")
dates = np.load(f"{DATA}/sample_dates.npy", allow_pickle=True)

idx = {
    "train": year_mask(dates, *config.TRAIN_YEARS),
    "val": year_mask(dates, *config.VAL_YEARS),
    "test": year_mask(dates, *config.TEST_YEARS),
}

model = VAEConv3D(
    input_dim=7, hidden_dim=256, latent_dim=128,
    input_shape=(11, config.TARGET_H, config.TARGET_W),
    kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1),
    apply_sigmoid=False,
).to(device)
model.load_state_dict(torch.load(f"{DATA}/vae_conus_L128_b01.pth", map_location=device))
model.eval()
print({k: len(v) for k, v in idx.items()})

# Loader Creation

In [ ]:
recon_df = {}
for period, i in idx.items():
    Xp, dp = X[i], dates[i]
    loader = make_analysis_loader(Xp, dp, batch_size=32)
    x, recon_x, z, dates_out = reconstruct_data(model, loader, device)
    if period == "test":
        recon_df[period] = {"x": x, "recon_x": recon_x, "z": z, "date_labels": dates_out}
    else:
        recon_df[period] = {"z": z, "date_labels": dates_out}
    print(f"{period}: z={z.shape}")
    del Xp, dp, loader, x, recon_x
    gc.collect()

# PCA

In [ ]:
# Fits on the train set, projects on all
recon_df = fit_latent_pca(recon_df)
fig = plot_scree(recon_df, n_show=128)
plt.show()

# t-SNE

In [ ]:
# t-SNE by train/val/test split
recon_df = compute_latent_tsne(recon_df, perplexity=50)
fig = plot_tsne_by_split(recon_df, perplexity=50)
plt.show()

In [ ]:
# t-SNE by season
from vae_analysis import plot_tsne_by_season

fig = plot_tsne_by_season(recon_df, perplexity=50)
plt.show()

# R-Squared

In [ ]:
from vae_analysis import calculate_r2_score

overall_r2, channel_r2 = calculate_r2_score(
    recon_df["test"]["x"], recon_df["test"]["recon_x"], SURFACE
)
print(f"overall R²: {overall_r2:.4f}")
for v, s in channel_r2.items():
    print(f"  {v}: {s:.4f}")

# GMM

In [ ]:
from sklearn.mixture import GaussianMixture

n_pca = 10
Xg = recon_df["test"]["pca"][:, :n_pca]

K_range = range(1, 11)
bic = {}
for K in K_range:
    gmm = GaussianMixture(n_components=K, covariance_type="diag",
        n_init=3, init_params="k-means++", random_state=42).fit(Xg)
    bic[K] = gmm.bic(Xg)

k_hat = min(bic, key=bic.get) # BIC is minimized
print(f"optimal K = {k_hat}")
for K, v in bic.items():
    print(f"  K={K}: BIC={v:.1f}")

# Combined Plot

In [ ]:
from vae_analysis import plot_combined_tsne_figure

fig = plot_combined_tsne_figure(recon_df, perplexity=50)
fig.savefig(f"{DATA}/figure4_tsne_gmm.png", dpi=300, bbox_inches="tight")

# Composite Plots

In [ ]:
from vae_analysis import plot_cluster_composites

domain = (config.LON_MIN, config.LON_MAX, config.LAT_MIN, config.LAT_MAX)
figs = plot_cluster_composites(recon_df, SURFACE, domain, period="test", perplexity=50, n_samples=100)
for var_name, fig in figs.items():
    fig.savefig(f"{DATA}/composite_{var_name}.png", dpi=300, bbox_inches="tight")